Perform text cleaning, perform lemmatization (any method), remove stop words (any method),
label encoding. Create representations using TF-IDF. Save outputs

In [2]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to /home/kedar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/kedar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /home/kedar/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw-1.4 to /home/kedar/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

### Pipeline 

In [5]:
import re
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer


In [8]:
df = pd.read_csv();


TypeError: read_csv() missing 1 required positional argument: 'filepath_or_buffer'

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)         # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:

# Lemmatization + stopword removal
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    return " ".join(tokens)

In [7]:
# Apply cleaning and preprocessing
df["cleaned_text"] = df["text"].astype(str).apply(clean_text)
df["processed_text"] = df["cleaned_text"].apply(preprocess_text)

# ----------------------------
# Label Encoding
# ----------------------------
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])

# Save label mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
pd.DataFrame(
    label_mapping.items(),
    columns=["original_label", "encoded_label"]
).to_csv("label_mapping.csv", index=False)

NameError: name 'df' is not defined

In [ ]:

# ----------------------------
# TF-IDF Vectorization
# ----------------------------
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_tfidf = tfidf.fit_transform(df["processed_text"])

# Convert TF-IDF to DataFrame
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

# ----------------------------
# Save outputs
# ----------------------------
df.to_csv("processed_text_data.csv", index=False)
tfidf_df.to_csv("tfidf_features.csv", index=False)

# Save vocabulary
pd.DataFrame(
    tfidf.get_feature_names_out(),
    columns=["feature"]
).to_csv("tfidf_vocabulary.csv", index=False)

print("✅ Text preprocessing, encoding, TF-IDF creation completed and saved.")
